In [15]:
import pandas as pd
import numpy as np
from astropy.table import Table, QTable
import astropy.units as u

from fits_utils import *

from qmostetc import SEDTemplate, QMostObservatory, Spectrum, L1DXU
from qmostetc.catalog import _split_magtype as split_magtype

In [2]:
template_4fs = Table.read('./../4M_templ_z1_00_extended.fits')
template_4fs

LAMBDA,FLUX_DENSITY
Angstrom,erg / (Angstrom s cm2)
float64,float64
5.0,2.328523
10.0,2.328523
15.0,2.328523
20.0,2.328523
25.0,2.328523
30.0,2.328523
35.0,2.328523
40.0,2.328523


In [3]:
spec = Spectrum(np.asarray(template_4fs['LAMBDA']) * u.Angstrom, 
                np.asarray(template_4fs['FLUX_DENSITY']) * u.erg / (u.cm**2 * u.s * u.Angstrom)
                )
spec

In [4]:
template = SEDTemplate(spec)#.to('erg / (nm m² s)'))
flux = template(20*u.ABmag, 'DECam.z')

qmost = QMostObservatory('hrs')  # high-resolution
obs = qmost(45*u.deg,  # airmass
            1.3*u.arcsec,  # seeing
            'gray')  # moon conditions

obs.set_target(flux, 'point')
tbl = obs.expose((5*60*60)*u.s)
tbl

wavelength,binwidth,efficiency,gain,target,sky,dark,ron,noise,arm
nm,nm,electron / ph,electron / adu,Angstrom electron / nm,electron,electron,electron,electron,
float64,float64,float64,float64,float64,float64,float64,float64,float64,str5
392.60692846030474,0.007922513014023025,0.1020396156178534,1.0514999999999999,182.48929072884124,82.60666374512797,15.61481875,6.208353296809067,12.45046433776717,blue
392.61485086465916,0.007922295694868353,0.10206980419587068,1.0514999999999999,182.54077638910744,82.52563926588108,15.61481875,6.208353296809067,12.447416853017812,blue
392.6227730516839,0.007922078354624773,0.10210015341074719,1.0514999999999999,182.59102026298007,83.04619618001001,15.61481875,6.208353296809067,12.468511042394278,blue
392.63069502135795,0.007921860993405971,0.10213065037771042,1.0514999999999999,182.64075593436695,86.37263553538992,15.61481875,6.208353296809067,12.601395975717397,blue
392.63861677366026,0.007921643611211948,0.10216126507272123,1.0514999999999999,182.69051229703396,93.86081137191901,15.61481875,6.208353296809067,12.895283324131443,blue
392.64653830856986,0.00792142620798586,0.10219198240460484,1.0514999999999999,182.74042811979854,98.02617274976919,15.61481875,6.208353296809067,13.055982727077616,blue
392.65445962606566,0.007921208783670863,0.10222281478523706,1.0514999999999999,182.79054518974544,91.98953816899252,15.61481875,6.208353296809067,12.822911607586128,blue
392.6623807261268,0.007920991338608019,0.10225372022098599,1.0514999999999999,182.84078937941987,84.421355699671,15.61481875,6.208353296809067,12.52453208888911,blue


In [13]:
tbl.write('./../pre_L1_spectrum.csv', format='csv')

In [ ]:
# np.save('/data2/home2/nguerrav/QSO_simpaqs/npy_files/etc_wavelength_grid.npy', 
#         np.asarray(tbl['wavelength']) * 10,  # in angstroms
#         allow_pickle=True)

In [ ]:
# tbl.write('./../pre_L1_spectrum.fits', overwrite=True)

In [20]:
dxu = L1DXU(qmost, tbl, (5*60*60)*u.s)
hdu_list = dxu.joined()
hdu_list[1].data

/home/nguerrav/miniconda3/envs/etc_4fs/lib/python3.12/site-packages/astropy/units/quantity.py:658: RuntimeWarning: invalid value encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)


FITS_rec([([3926.1 , 3926.15, 3926.2 , ..., 6789.8 , 6789.85, 6789.9 ], [ 1.34750088e-17,  1.02499829e-17,  8.19101953e-18, ...,  5.34170327e-18,  5.86329110e-18,  6.46113465e-18], [7.29707800e-18, 7.29469655e-18, 7.29424573e-18, ..., 3.06008726e-18, 3.01009660e-18, 2.96840690e-18], [0, 0, 0, ..., 0, 0, 0], [5.19378170e-17, 4.86855579e-17, 4.66372370e-17, ..., 3.64465406e-17, 3.55916594e-17, 3.50587036e-17], [7.29707800e-18, 7.29469655e-18, 7.29424573e-18, ..., 3.06008726e-18, 3.01009660e-18, 2.96840690e-18])],
         dtype=(numpy.record, [('WAVE', '<f4', (33775,)), ('FLUX', '<f4', (33775,)), ('ERR_FLUX', '<f4', (33775,)), ('QUAL', '<i4', (33775,)), ('FLUX_NOSS', '<f4', (33775,)), ('ERR_FLUX_NOSS', '<f4', (33775,))]))

In [18]:
Table(hdu_list[1].data[:])

WAVE,FLUX,ERR_FLUX,QUAL,FLUX_NOSS,ERR_FLUX_NOSS
float32[33775],float32[33775],float32[33775],int32[33775],float32[33775],float32[33775]
3926.1 .. 6789.9,2.240988e-17 .. 8.782292e-18,7.297078e-18 .. 2.968407e-18,0 .. 0,6.0872684e-17 .. 3.7379862e-17,7.297078e-18 .. 2.968407e-18


In [21]:
# hdu_list.writeto('./../L1_spectrum.fits', overwrite=True)
l1_spec = Table.read('./../L1_spectrum.fits')
l1_spec

WAVE,FLUX,ERR_FLUX,QUAL,FLUX_NOSS,ERR_FLUX_NOSS
Angstrom,erg / (Angstrom s cm2),erg / (Angstrom s cm2),,erg / (Angstrom s cm2),erg / (Angstrom s cm2)
float32[33775],float32[33775],float32[33775],int32[33775],float32[33775],float32[33775]
3926.10 .. 6789.90,7.40116e-18 .. 6.37232e-18,1.10097e-18 .. 4.95474e-19,0 .. 0,4.58640e-17 .. 3.49699e-17,1.10097e-18 .. 4.95474e-19


In [30]:
l1_spec

WAVE,FLUX,ERR_FLUX,QUAL,FLUX_NOSS,ERR_FLUX_NOSS
Angstrom,erg / (Angstrom s cm2),erg / (Angstrom s cm2),,erg / (Angstrom s cm2),erg / (Angstrom s cm2)
float32[33775],float32[33775],float32[33775],int32[33775],float32[33775],float32[33775]
3926.10 .. 6789.90,7.40116e-18 .. 6.37232e-18,1.10097e-18 .. 4.95474e-19,0 .. 0,4.58640e-17 .. 3.49699e-17,1.10097e-18 .. 4.95474e-19


In [31]:
l1_spec = Table(hdu_list[1].data[:])
l1_spec

WAVE,FLUX,ERR_FLUX,QUAL,FLUX_NOSS,ERR_FLUX_NOSS
float32[33775],float32[33775],float32[33775],int32[33775],float32[33775],float32[33775]
3926.1 .. 6789.9,1.3475009e-17 .. 6.4611346e-18,7.297078e-18 .. 2.968407e-18,0 .. 0,5.1937817e-17 .. 3.5058704e-17,7.297078e-18 .. 2.968407e-18


In [33]:
l1_spec['WAVE'][0]

array([3926.1 , 3926.15, 3926.2 , ..., 6789.8 , 6789.85, 6789.9 ],
      dtype=float32)

In [34]:
n = len(l1_spec[l1_spec.colnames[0]][0])
l1_spec_flat = Table({col: l1_spec[col][0] for col in l1_spec.colnames if l1_spec[col][0].ndim == 1 and len(l1_spec[col][0]) == n})
l1_spec_flat

WAVE,FLUX,ERR_FLUX,QUAL,FLUX_NOSS,ERR_FLUX_NOSS
float32,float32,float32,int32,float32,float32
3926.1,1.3475009e-17,7.297078e-18,0,5.1937817e-17,7.297078e-18
3926.15,1.0249983e-17,7.2946965e-18,0,4.8685558e-17,7.2946965e-18
3926.2,8.1910195e-18,7.294246e-18,0,4.6637237e-17,7.294246e-18
3926.25,1.2106958e-17,7.300644e-18,0,5.0699015e-17,7.300644e-18
3926.3,1.1086965e-17,7.334124e-18,0,5.036299e-17,7.334124e-18
3926.35,7.9404136e-18,7.385893e-18,0,4.8269913e-17,7.385893e-18
3926.4,1.14585575e-17,7.493407e-18,0,5.3978812e-17,7.493407e-18
3926.45,1.2653769e-17,7.576376e-18,0,5.689145e-17,7.576376e-18
3926.5,1.1378758e-17,7.634171e-18,0,5.683056e-17,7.634171e-18


In [35]:
type(l1_spec_flat)

astropy.table.table.Table

In [36]:
l1_spec_flat.write('./../L1_spectrum.csv', format='csv')